# modular-diffusion: what each noise band of Stable Diffusion 1.5 needs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.8.0/colab/band_needs.ipynb)

The capture library makes one smaller U-Net (a band engine) per stretch of the noise range from Stable Diffusion 1.5 by removing blocks in mirrored pairs, every number copied from the model (`geolip.modular_diffusion.capture`). Before any engine is trained, this notebook measures, with no training at all, what each of the three bands (clean, middle, noisy) needs: the born read (the mean squared difference from the model's own answer on that band) of every single block removal and of the shapes base, small and tiny; how much each band needs the prompt (the final latent of 50 deterministic DDIM steps with the prompt replaced, inside one band only, by the empty prompt or by another row's prompt); and the measured cost of every shape. Every number is read on two probe seeds, and the controls must hold exactly: the model's own copy reads 0, a random engine reads far above an inherited one, the reference sampled again changes by 0.

1. install at a release tag and print the environment;
2. the model at a pinned revision, 256 captioned COCO val2017 pictures with their content hash, their latents from the model's own image encoder, a smoke run (shipped, so the token and the upload are checked before the long run) and a measured projection of the run's minutes on this card;
3. the run, resumable unit by unit (run the cell again after a stop), its picture list, the upload, the tables; then the machine is given back.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.8.0"
import os, sys, time, json, gc, hashlib
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 8, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/band_needs"          # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

The model comes from the Hub at a pinned revision (the U-Net, the text encoder and its tokenizer, the image encoder) and the pictures from one parquet file of COCO val2017 with five captions each (the first is used). The content hash covers every picture as the 512 x 512 RGB array the image encoder sees and its caption, so a later run can check it has the same data. The latents are written once (skipped when they are already there). The smoke run reads the tiny random model with the job's own code on this card and is shipped at once; the timing run measures forwards of the real model on this card and projects the minutes of each read.

In [ ]:
# C2 - the model, the pictures and their content hash, the latents, a smoke run, a measured projection
from huggingface_hub import snapshot_download, hf_hub_download
from geolip.modular_diffusion import hub
from geolip.modular_diffusion.capture.data import center_crop, encode_shards, parquet_pictures, read_index
from geolip.modular_diffusion.capture.teacher import load_teacher
from geolip.modular_diffusion.experiments import band_needs
TEACHER = f"{OUT}/sd15"
snapshot_download(repo_id="stable-diffusion-v1-5/stable-diffusion-v1-5", revision="451f4fe16113bff5a5d2269ed5ad43b0592e9a14",
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)
PARQUET = hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset", revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff",
                          filename="data/val-00000-of-00002.parquet", local_dir=f"{OUT}/coco")
N = 256
pictures, captions, meta = parquet_pictures(PARQUET, N)

def content_hash(pictures, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()

EXPECTED_CONTENT_HASH = ""
HASH = content_hash(pictures, captions)
print("content hash", HASH, "| expected", EXPECTED_CONTENT_HASH or "(not filled)")
assert not EXPECTED_CONTENT_HASH or HASH == EXPECTED_CONTENT_HASH, "the pictures are not the expected data"
LICENSES = {}
for m in meta:
    LICENSES[str(m.get("license"))] = LICENSES.get(str(m.get("license")), 0) + 1
print("pictures per license id:", LICENSES)
LATENTS = f"{OUT}/latents"
if not (os.path.exists(f"{LATENTS}/index.json") and sum(read_index(LATENTS)["counts"]) == N):
    teacher = load_teacher(TEACHER, device="cuda", dtype=torch.float32, vae=True)
    encode_shards(pictures, captions, LATENTS, teacher.vae, teacher.tokenize, shard_size=64, batch=16, size=512, device="cuda")
    del teacher
    gc.collect()
    torch.cuda.empty_cache()
print("latent shards:", read_index(LATENTS)["counts"])
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
EXTRA = {}                                                      # more arguments for the timing and the run (empty in a real run)

# the smoke run, its upload, and the measured projection
DEVICE = "cuda"
SIZES = dict(seeds="0,1", rows=64, per_band=4, prompts=128, group=32, chunk=16, steps=50, guidance=7.5, cost_batch=16, cost_repeats=3)
smoke = band_needs.main(tiny=1, device=DEVICE, out=f"{OUT}/smoke", rows=4, prompts=4, group=2, chunk=2, per_band=2, steps=6,
                        cost_batch=2, cost_repeats=1)
assert all(v == 0.0 for s in smoke["controls"] for v in smoke["controls"][s]["full"].values()), "the copy does not read 0"
assert all(smoke["r3"][s]["repeat"] == 0.0 for s in smoke["r3"]), "the sampler does not repeat itself"
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    assert hub.ship(f"{OUT}/smoke", REPO, "modular_diffusion/capture/band_needs/_smoke", "band needs: smoke run"), \
        "the smoke folder was not shipped: check the token and the repository before the long run"
timing = band_needs.main(variant="timing", teacher=TEACHER, data=LATENTS, device=DEVICE, out=f"{OUT}/timing", **{**SIZES, **EXTRA})
m = timing["minutes"]
print(f"projected minutes on this card: born reads with controls {m['reads']:.1f}, cost {m['r4']:.1f}, prompt read {m['r3']:.1f}; "
      f"total {m['total']:.1f}")

The run reads the model on this card unit by unit; each finished unit is written under `RUN/parts` at once, so after a stop the same cell continues where it stopped. The picture list (row, COCO id, license id, file name and the content hash; no pictures and no caption text) is written into the run folder, the folder is shipped to `REPO` with the token placed in C2, the tables are printed, and the machine is given back (`RELEASE`) only when the upload returned, since an idle machine keeps spending.

In [ ]:
# C3 - the run (resumable unit by unit), its picture list, the upload, the tables, the machine given back
RELEASE = True                       # give the machine back when the run is shipped
RUNSIZE = {**SIZES, **EXTRA}
NAME = (f"sd15_rows{RUNSIZE['rows']}x{RUNSIZE['per_band']}_prompts{RUNSIZE['prompts']}_steps{RUNSIZE['steps']}"
        f"_g{RUNSIZE['guidance']}_seeds{RUNSIZE['seeds'].replace(',', '')}")
RUN = f"{OUT}/runs/{NAME}"
os.makedirs(RUN, exist_ok=True)
with open(f"{RUN}/pictures.json", "w", encoding="utf-8") as fh:
    json.dump({"source": {"repo": "lmms-lab-encoder/COCO-Caption2017", "revision": "3bdd5827e243cc3084ac69a1111e69c3ab9193ff", "file": "data/val-00000-of-00002.parquet", "rows": N},
               "content_hash": HASH, "licenses": LICENSES,
               "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license"), "file_name": m.get("file_name")}
                            for i, m in enumerate(meta)]}, fh, indent=1)
result = band_needs.main(teacher=TEACHER, data=LATENTS, device=DEVICE, out=RUN, **RUNSIZE)
print(open(f"{RUN}/result.md", encoding="utf-8").read())
shipped = hub.ship(RUN, REPO, f"modular_diffusion/capture/band_needs/{NAME}", f"band needs: {NAME}") if REPO else None
print("shipped:", shipped)
if RELEASE and REPO and shipped:
    from google.colab import runtime
    runtime.unassign()